# 03️⃣ Modeling

Train a LightGBM regressor to forecast tomorrow's close price return using stationary, scale-invariant features. Split data temporally.

In [1]:
import os, pathlib, pandas as pd, numpy as np, joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
%matplotlib inline


In [2]:
NOTEBOOK_DIR = pathlib.Path(os.path.abspath('') if '__file__' not in locals() else os.path.dirname(__file__))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PARQUET = PROJECT_ROOT / 'data' / 'parquet'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PARQUET / 'feature_engineered.parquet')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Symbol', 'Date'])

# Create stationary features (avoid scale differences like TCS vs ONGC)
df['MA20_Ratio'] = df['Close'] / (df['MA20'] + 1e-9) - 1.0
df['MA50_Ratio'] = df['Close'] / (df['MA50'] + 1e-9) - 1.0
df['EMA20_Ratio'] = df['Close'] / (df['EMA20'] + 1e-9) - 1.0
df['Momentum_Ratio'] = df['Momentum'] / (df['Close'] - df['Momentum'] + 1e-9)
df['MACD_Ratio'] = df['MACD'] / (df['Close'] + 1e-9)
df['Signal_Ratio'] = df['Signal_Line'] / (df['Close'] + 1e-9)
df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / (df['BB_Middle'] + 1e-9)
df['BB_Pos'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'] + 1e-9)

# Target: Tomorrow's return (grouped by Symbol to avoid look-ahead or cross-symbol leakage)
df['Target_Return'] = df.groupby('Symbol')['Return'].shift(-1)
df = df.dropna()

features = [
    'Return', 'MA20_Ratio', 'MA50_Ratio', 'EMA20_Ratio',
    'Volatility', 'Momentum_Ratio', 'RSI', 'MACD_Ratio',
    'Signal_Ratio', 'BB_Width', 'BB_Pos'
]
X = df[features]
y = df['Target_Return']

# Strict time-based split: Train before 2016-01-01, Test on 2016-01-01 to 2021-04-30
train_mask = df['Date'] < '2016-01-01'
test_mask = df['Date'] >= '2016-01-01'

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')
print('Training LightGBM model...')
model = LGBMRegressor(n_estimators=100, learning_rate=0.01, max_depth=5, num_leaves=15, random_state=42, verbose=-1)
model.fit(X_train, y_train)

# Evaluate on out-of-sample test set
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Directional Accuracy
same_sign = np.sign(y_test.values) == np.sign(y_pred)
accuracy_dir = np.mean(same_sign)

print(f'Test Performance Out-of-Sample (2016-2021):\nMAE: {mae:.6f}\nRMSE: {rmse:.6f}\nR2: {r2:.6f}\nDirectional Accuracy: {accuracy_dir:.2%}')

# Save metrics
with open(OUTPUT_DIR / 'model_metrics.txt', 'w') as f:
    f.write(f'MAE: {mae:.6f}\nRMSE: {rmse:.6f}\nR2: {r2:.6f}\nDirectionalAccuracy: {accuracy_dir:.2%}')

# Save model
model_path = MODEL_DIR / 'gbr_model.pkl'
joblib.dump(model, model_path)
print(f'Model saved to {model_path}')

# Save dataset with predictions
df['Predicted_Return'] = model.predict(X)
df.to_parquet(DATA_PARQUET / 'modeled_data.parquet', index=False)


Train samples: 55462, Test samples: 64533
Training LightGBM model...


Test Performance Out-of-Sample (2016-2021):
MAE: 0.014577
RMSE: 0.023570
R2: -0.004126
Directional Accuracy: 50.17%
Model saved to C:\Users\sajal\cult project\netflix\models\gbr_model.pkl
